# Similarity-aware Label Smoothing

In [77]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset Preparation

### MNIST

In [68]:
mnist_train_ds_raw = datasets.MNIST(root="./data/mnist", train=True, download=True, transform=transforms.ToTensor())

all_imgs = torch.stack([img for img, _ in mnist_train_ds_raw], dim = 0)
ds_mean = all_imgs.mean(dim=(0, 2, 3))
ds_std = all_imgs.std(dim=(0, 2, 3))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((ds_mean, ), (ds_std, ))
])

mnist_train_ds = datasets.MNIST(root="./data/mnist", train=True, download=True, transform=transform)
mnist_test_ds = datasets.MNIST(root="./data/mnist", train=False, transform=transform)

train_loader = DataLoader(mnist_train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(mnist_test_ds, batch_size=256)

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.68MB/s]


### CIFAR-100

In [ ]:
cifar_train_ds_raw = datasets.CIFAR100(root="./data/cifar", train=True, download=True, transform=transforms.ToTensor())

all_imgs = torch.stack([img for img, _ in cifar_train_ds_raw], dim = 0)
ds_mean = all_imgs.mean(dim=(0, 2, 3))
ds_std = all_imgs.std(dim=(0, 2, 3))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=ds_mean.tolist(), std=ds_std.tolist())
])

cifar_train_ds = datasets.CIFAR100(root="./data/cifar", train=True, download=True, transform=transform)
cifar_test_ds = datasets.CIFAR100(root="./data/cifar", train=False, transform=transform)

train_loader = DataLoader(cifar_train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(cifar_test_ds, batch_size=256)

100%|██████████| 169M/169M [00:18<00:00, 9.01MB/s] 
/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


tensor([0.5071, 0.4866, 0.4409]) tensor([0.2673, 0.2564, 0.2762])


## Modeld

### MNIST

In [70]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


### CIFAR-100

In [78]:
def RESNET18(pretrained=False):
    model = models.resnet18(pretrained=pretrained)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)

    return model

## Training Utils

### Accuracy Measure

In [71]:
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim = 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

### Label Smoothing

In [64]:
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

## Training Loop

In [72]:
def train(model, loader, optimizer, epochs = 10, epsilon = 0.1):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)

            targets = uniform_smooth_labels(y, num_classes=10, epsilon=epsilon)

            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss/len(loader):.4f} | Test Acc: {acc:.4f}")

## RUN Experiments

In [ ]:
model = RESNET18().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

train(model, train_loader, optimizer, epochs=10, epsilon=0)

/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


RuntimeError: Class values must be smaller than num_classes.